# DataFrame Concepts — From Scratch to Pandas

**Goal:** Understand what a DataFrame *really is* by building one ourselves using Python dictionaries and loops, then comparing our manual work with Pandas abstractions.

We will:
1. Read a CSV file using only the built-in `csv` module
2. Store the data in a dictionary-of-lists (the same internal structure Pandas uses)
3. Perform common operations (filter, aggregate, sort) with plain loops
4. Show the same operations in Pandas — one-liners that replace dozens of lines
5. Visualize the results

## 1 — Read CSV the Hard Way (No Pandas)

In [1]:
import csv

csv_path = "../PY200/src/excel_100_practice_workbooks.csv"

# ---------- Step 1: Read every row into a list of dictionaries ----------
with open(csv_path, newline="") as f:
    reader = csv.DictReader(f)          # each row → OrderedDict
    rows = list(reader)                 # materialise into a list

# Quick look at the first two records
for row in rows[:2]:
    print(row)
print(f"\nTotal rows read: {len(rows)}")

{'Match Date': '07-Jan-2026', 'Team': 'Falcons', 'Opponent': 'Comets', 'Venue': 'River Court', 'Player': 'Zara', 'Grade': 'Grade 6', 'Minutes Played': '22', 'Points Scored': '29', 'Assists': '1', 'Rebounds': '3', 'Fouls': '4', 'Ticket Sales ($)': '$2,628', 'Coach': 'Coach Maya', 'Match Result': 'Win'}
{'Match Date': '09-Jan-2026', 'Team': 'Falcons', 'Opponent': 'Comets', 'Venue': 'Central Dome', 'Player': 'Zara', 'Grade': 'Grade 5', 'Minutes Played': '35', 'Points Scored': '12', 'Assists': '11', 'Rebounds': '14', 'Fouls': '1', 'Ticket Sales ($)': '$2,739', 'Coach': 'Coach Maya', 'Match Result': 'Loss'}

Total rows read: 44


### 1.1 — Build a Column-Oriented Dictionary (our "DataFrame")

A Pandas DataFrame stores data **column-wise** — each column is a contiguous array.  
We can mimic this with a `dict[str, list]`:

In [2]:
# ---------- Step 2: Convert row-oriented list → column-oriented dict ----------
columns = list(rows[0].keys())       # grab column names from the first row

# Initialise an empty list for each column
data = {col: [] for col in columns}

# Fill column lists row by row
for row in rows:
    for col in columns:
        data[col].append(row[col])

# Show structure
print("Columns:", columns)
print(f"\nFirst 5 values in 'Player': {data['Player'][:5]}")
print(f"First 5 values in 'Points Scored': {data['Points Scored'][:5]}")

Columns: ['Match Date', 'Team', 'Opponent', 'Venue', 'Player', 'Grade', 'Minutes Played', 'Points Scored', 'Assists', 'Rebounds', 'Fouls', 'Ticket Sales ($)', 'Coach', 'Match Result']

First 5 values in 'Player': ['Zara', 'Zara', 'Lily', 'Aarav', 'Ryan']
First 5 values in 'Points Scored': ['29', '12', '12', '9', '8']


### 1.2 — Clean / Cast Data Types

CSV values are always **strings**. We need to convert numeric columns ourselves.  
Pandas does this automatically with `read_csv` — here we do it by hand:

In [3]:
# ---------- Step 3: Cast numeric columns from strings → int / float ----------
int_cols = ["Minutes Played", "Points Scored", "Assists", "Rebounds", "Fouls"]

for col in int_cols:
    data[col] = [int(v) for v in data[col]]

# Ticket Sales has "$" and commas — strip them first
data["Ticket Sales ($)"] = [
    int(v.replace("$", "").replace(",", ""))
    for v in data["Ticket Sales ($)"]
]

# Verify types
print(f"Points Scored (first 5): {data['Points Scored'][:5]}  → type: {type(data['Points Scored'][0])}")
print(f"Ticket Sales  (first 5): {data['Ticket Sales ($)'][:5]}  → type: {type(data['Ticket Sales ($)'][0])}")

ValueError: invalid literal for int() with base 10: ''

---
## 2 — Common Operations: Loops vs Pandas

Now that we have our manual "DataFrame" (`data` dict), let's perform common tasks **side-by-side** with Pandas.

In [ ]:
# Load the same CSV with Pandas for comparison
import pandas as pd

df = pd.read_csv(csv_path)
df.head()

### 2.1 — Filtering Rows

**Task:** Get all rows where `Team == "Falcons"` and `Points Scored > 15`

In [ ]:
# ===== LOOP VERSION =====
filtered_loop = []
for i in range(len(data["Team"])):
    if data["Team"][i] == "Falcons" and data["Points Scored"][i] > 15:
        # Reconstruct a row dict from column lists
        filtered_loop.append({col: data[col][i] for col in columns})

print(f"Loop: Found {len(filtered_loop)} rows")
for row in filtered_loop[:3]:
    print(f"  {row['Player']:>6} | {row['Points Scored']:>3} pts | {row['Match Date']}")

In [ ]:
# ===== PANDAS VERSION =====
# One line replaces the entire loop above
filtered_pd = df[(df["Team"] == "Falcons") & (df["Points Scored"] > 15)]
print(f"Pandas: Found {len(filtered_pd)} rows")
filtered_pd[["Player", "Points Scored", "Match Date"]].head()

### 2.2 — Aggregation (Group By)

**Task:** Total points scored per team

In [ ]:
# ===== LOOP VERSION =====
team_points = {}                        # {team_name: total_points}
for i in range(len(data["Team"])):
    team = data["Team"][i]
    pts  = data["Points Scored"][i]
    if team in team_points:
        team_points[team] += pts
    else:
        team_points[team] = pts

print("Loop — Total points per team:")
for team, pts in sorted(team_points.items(), key=lambda x: x[1], reverse=True):
    print(f"  {team:<10} {pts}")

In [ ]:
# ===== PANDAS VERSION =====
# groupby + sum replaces the entire loop + dictionary accumulation
df.groupby("Team")["Points Scored"].sum().sort_values(ascending=False)

### 2.3 — Sorting

**Task:** Sort all rows by `Points Scored` descending, show top 5

In [ ]:
# ===== LOOP VERSION =====
# Build index list, sort it by Points Scored, then extract top 5
indices = list(range(len(data["Points Scored"])))
indices.sort(key=lambda i: data["Points Scored"][i], reverse=True)

print("Loop — Top 5 scorers:")
for rank, i in enumerate(indices[:5], 1):
    print(f"  {rank}. {data['Player'][i]:<8} ({data['Team'][i]:<10}) — {data['Points Scored'][i]} pts")

In [ ]:
# ===== PANDAS VERSION =====
df.nlargest(5, "Points Scored")[["Player", "Team", "Points Scored"]]

### 2.4 — Adding a Computed Column

**Task:** Create a "Performance Score" = `Points Scored + (Assists × 2) + Rebounds`

In [ ]:
# ===== LOOP VERSION =====
data["Performance Score"] = []
for i in range(len(data["Points Scored"])):
    score = data["Points Scored"][i] + (data["Assists"][i] * 2) + data["Rebounds"][i]
    data["Performance Score"].append(score)

print("Loop — First 5 Performance Scores:", data["Performance Score"][:5])

In [ ]:
# ===== PANDAS VERSION =====
# Vectorized arithmetic — no loop needed, operates on entire columns at once
df["Performance Score"] = df["Points Scored"] + (df["Assists"] * 2) + df["Rebounds"]
df[["Player", "Team", "Points Scored", "Assists", "Rebounds", "Performance Score"]].head()

---
## 3 — Visualization: Seeing the Difference

Charts make it obvious why Pandas is worth learning — `df.plot()` is one line vs building lists by hand.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- LEFT: Bar chart using our manual dict (loop-built data) ---
ax1 = axes[0]
teams = list(team_points.keys())
totals = [team_points[t] for t in teams]
# Sort for a clean chart
sorted_pairs = sorted(zip(teams, totals), key=lambda x: x[1], reverse=True)
teams_sorted = [p[0] for p in sorted_pairs]
totals_sorted = [p[1] for p in sorted_pairs]

ax1.barh(teams_sorted, totals_sorted, color="steelblue")
ax1.set_xlabel("Total Points Scored")
ax1.set_title("Loop Version (manual dict)")
ax1.invert_yaxis()

# --- RIGHT: Same chart in one line with Pandas ---
ax2 = axes[1]
df.groupby("Team")["Points Scored"].sum().sort_values().plot.barh(
    ax=ax2, color="darkorange"
)
ax2.set_xlabel("Total Points Scored")
ax2.set_title("Pandas Version (one-liner)")

plt.tight_layout()
plt.show()

### 3.1 — Player Performance Scatter Plot

In [ ]:
# ===== LOOP VERSION — Scatter: Minutes Played vs Performance Score =====
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
# Build colour map per team manually
unique_teams = list(set(data["Team"]))
color_map = dict(zip(unique_teams, plt.cm.tab10.colors[:len(unique_teams)]))

for i in range(len(data["Team"])):
    ax1.scatter(
        data["Minutes Played"][i],
        data["Performance Score"][i],
        color=color_map[data["Team"][i]],
        alpha=0.7, edgecolors="k", linewidths=0.3
    )
# Manual legend
for team, color in color_map.items():
    ax1.scatter([], [], color=color, label=team)
ax1.legend(fontsize=8)
ax1.set_xlabel("Minutes Played")
ax1.set_ylabel("Performance Score")
ax1.set_title("Loop Version (manual scatter)")

# ===== PANDAS VERSION — same chart =====
ax2 = axes[1]
for team, group in df.groupby("Team"):
    ax2.scatter(group["Minutes Played"], group["Performance Score"],
                label=team, alpha=0.7, edgecolors="k", linewidths=0.3)
ax2.legend(fontsize=8)
ax2.set_xlabel("Minutes Played")
ax2.set_ylabel("Performance Score")
ax2.set_title("Pandas Version (groupby scatter)")

plt.tight_layout()
plt.show()

### 3.2 — Win/Loss Distribution per Team

In [ ]:
# ===== LOOP VERSION — Count wins & losses per team =====
win_loss = {}   # {team: {"Win": count, "Loss": count}}
for i in range(len(data["Team"])):
    team   = data["Team"][i]
    result = data["Match Result"][i]
    if team not in win_loss:
        win_loss[team] = {"Win": 0, "Loss": 0}
    if result in win_loss[team]:
        win_loss[team][result] += 1

print("Loop — Win/Loss counts:")
for team in sorted(win_loss):
    w, l = win_loss[team]["Win"], win_loss[team]["Loss"]
    print(f"  {team:<10}  W={w}  L={l}")

In [ ]:
# ===== PANDAS VERSION — crosstab + stacked bar =====
ct = pd.crosstab(df["Team"], df["Match Result"])
ct.plot.bar(stacked=True, color=["#e74c3c", "#2ecc71"], figsize=(8, 4))
plt.title("Win / Loss per Team")
plt.ylabel("Number of Matches")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 4 — Key Takeaways

| Task | Loop (manual dict) | Pandas |
|---|---|---|
| **Read CSV** | `csv.DictReader` → convert types manually | `pd.read_csv()` — auto-detects types |
| **Filter rows** | Loop + `if` + rebuild row dicts | Boolean indexing: `df[df["col"] > x]` |
| **Group & aggregate** | Dict accumulator + loop | `df.groupby("col").sum()` |
| **Sort** | Build index list + `sort(key=...)` | `df.sort_values()` / `df.nlargest()` |
| **New column** | Loop + append per element | Vectorized: `df["new"] = df["a"] + df["b"]` |
| **Visualize** | Build lists → `plt.bar(...)` | `df.plot.bar()` — labels auto-generated |

**Bottom line:** A DataFrame is just a dictionary of aligned lists/arrays with a rich API on top. Understanding the manual version helps you reason about what Pandas does under the hood — and appreciate *why* it exists.